# Analyse ROMY Events - FUR vs ADR

In [ ]:
import os
import obspy as obs
import matplotlib.pyplot as plt
import numpy as np

from obspy.clients.fdsn import Client
from functions.read_sds import __read_sds


In [3]:
1/80

0.0125

In [ ]:
if os.uname().nodename == 'lighthouse':
    root_path = '/home/andbro/'
    data_path = '/home/andbro/kilauea-data/'
    archive_path = '/home/andbro/freenas/'
    bay_path = '/home/andbro/bay200/'
    lamont_path = '/home/andbro/lamont/'
elif os.uname().nodename == 'kilauea':
    root_path = '/home/brotzer/'
    data_path = '/import/kilauea-data/'
    archive_path = '/import/freenas-ffb-01-data/'
    bay_path = '/import/ontap-ffb-bay200/'
    lamont_path = '/lamont/'
elif os.uname().nodename in ['lin-ffb-01', 'ambrym', 'hochfelln']:
    root_path = '/home/brotzer/'
    data_path = '/import/kilauea-data/'
    archive_path = '/import/freenas-ffb-01-data/'
    bay_path = '/import/ontap-ffb-bay200/'
    lamont_path = '/lamont/'

## Configurations

In [ ]:
config = {}

# output path for figures
config['path_to_figs'] = data_path+"romy_events/figures/"

# path to data archive
config['path_to_data'] = archive_path+"temp_archive/"
config['path_to_bay'] = bay_path+"mseed_online/archive/"

# Event Morocco
config['event_name'] = "Morocco"
config['tbeg'] = obs.UTCDateTime("2023-09-08 22:00")
config['tend'] = obs.UTCDateTime("2023-09-08 23:00")
dt1, dt2 = 900, 1200

# specfiy seismometer
config['seis'] = "FUR"


config['Client'] = Client("USGS")

# ROMY coordinates
config['sta_lon'] = 11.275501
config['sta_lat'] = 48.162941

baz = 228.39

In [ ]:
def find_minimum(arr1, arr2, lagmin=-100, lagmax=100, scalemin=0.1, scalemax=2, dlag=1, dscale=0.1, plot=True):

    import matplotlib.pyplot as plt
    from numpy import unravel_index
    from functions.variance_reduction import __variance_reduction

    dshft = dlag
    shifts = np.arange(lagmin, lagmax+dshft, dshft)

    dfac = dscale
    factors = np.arange(scalemin, scalemax+dfac, dfac)

    Ns, Nf = len(shifts), len(factors)
    vr = np.zeros((Ns, Nf))

    for i, s in enumerate(shifts):
        for j, f in enumerate(factors):

            _arr1 = f*np.roll(arr1, s)

            vr[i, j] = __variance_reduction(arr2, arr2 - _arr1)

    imax, jmax = unravel_index(vr.argmax(), vr.shape)

    vr_max, s_max, f_max = round(vr[imax, jmax], 1), round(shifts[imax], 0), round(factors[jmax], 2)

    if plot:

        fig = plt.figure()

        cm = plt.pcolormesh(shifts,
                            factors,
                            vr[:-1, :-1].T,
                            rasterized=True,
                            cmap=plt.colormaps.get("seismic"),
                            vmin=-100, vmax=100,
                           )

        plt.scatter(s_max, f_max, color="black", s=15, edgecolor="w")

        plt.text(0, 0.99, f"{s_max}, {f_max}, {vr_max}%", ha='center', va='top', color="w")

        plt.xlabel("Shifts (samples)")
        plt.ylabel("Scaling Factor")

        cb = plt.colorbar(cm)
        cb.set_label("Variance Reduction (%)")

    return s_max, f_max, vr_max

## Load Data

In [ ]:
st0 = obs.Stream()

### Add ROMY Data

In [ ]:
st0 += __read_sds(config['path_to_data'], "BW.ROMY.30.BJ*", config['tbeg'], config['tend'])

### Add ADR Data

In [ ]:
# st0 += __read_sds(config['path_to_data'], "BW.ROMY.22.BJ*", config['tbeg'], config['tend'])

In [ ]:
fur = __read_sds(config['path_to_bay'], "GR.FUR..BH*", config['tbeg'], config['tend'])

fur_inv = obs.read_inventory(data_path+"stationxml_ringlaser/station_GR_FUR.xml")

fur = fur.remove_response(fur_inv)

st0 += fur.copy()

In [ ]:
st0 = st0.trim(config['tbeg']+dt1, config['tend']-dt2)

In [ ]:
st0 = st0.rotate('NE->RT', back_azimuth=baz)


In [ ]:
print(st0)

st0.plot(equal_scale=False);

## Filter

In [ ]:

fmin, fmax = 0.02, 0.1

acc = st0.select(station="FUR", channel="*H*").copy()
rot = st0.select(station="ROMY", channel="*J*").copy()

for tr in rot:
    tr.data *= 3500

# for tr in rot:
#     if "Z" in tr.stats.channel:
#         tr.data = np.roll(tr.data, 10)
#     if "N" in tr.stats.channel:
#         tr.data = np.roll(tr.data, 15)
#     if "E" in tr.stats.channel:
#         tr.data = np.roll(tr.data, 16)


acc = acc.detrend("demean").taper(0.05).filter("bandpass", freqmin=fmin, freqmax=fmax, corners=4, zerophase=True);
rot = rot.detrend("demean").taper(0.05).filter("bandpass", freqmin=fmin, freqmax=fmax, corners=4, zerophase=True);


In [ ]:
rot.plot();
acc.plot();

In [ ]:
arr1 = rot.select(component="Z")[0].data
arr2 = acc.select(component="T")[0].data

smax_z, fmax_z, vrmax_z = find_minimum(arr1, arr2, scalemin=-10, scalemax=10, dscale=1, plot=True)

In [ ]:
arr1 = rot.select(component="T")[0].data
arr2 = acc.select(component="Z")[0].data

smax_n, fmax_n, vrmax_n = find_minimum(arr1, arr2, scalemin=-10, scalemax=10, dscale=0.5, plot=True)

### Plot Comparison

In [ ]:
from functions.get_octave_bands import __get_octave_bands

vals_l = {}
vals_r = {}

fl, fu, fc = __get_octave_bands(0.01, 1.0, faction_of_octave=6)

dt = st0[0].stats.delta

for f1, f2, fc in zip(fl, fu, fc):

    acc = st0.copy().select(station="FUR", channel="*H*")
    rot = st0.copy().select(station="ROMY", channel="*J*")

    for tr in rot:
        tr.data *= 3000

    acc = acc.detrend("demean").taper(0.01)
    acc = acc.filter("bandpass", freqmin=f1, freqmax=f2, corners=4, zerophase=True);
    rot = rot.detrend("demean").taper(0.01)
    rot = rot.filter("bandpass", freqmin=f1, freqmax=f2, corners=4, zerophase=True);

    arr1 = rot.select(component="Z")[0].data
    arr2 = acc.select(component="T")[0].data

    vals_l[fc] = find_minimum(arr1, arr2, scalemax=10, dscale=0.5, lagmin=-150, lagmax=150, plot=False)
    plt.show();
    arr1 = rot.select(component="T")[0].data
    arr2 = acc.select(component="Z")[0].data

    vals_r[fc] = find_minimum(arr1, arr2, scalemax=10, dscale=0.1, lagmin=-150, lagmax=150, plot=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))

for x in vals_l.keys():
    ax[0].scatter(x, vals_l[x][0]*dt, s=abs(vals_l[x][2]), c="tab:blue", zorder=2)
    ax[0].set_xscale("log")
    ax[0].set_title("Love")

for x in vals_r.keys():
    ax[1].scatter(x, vals_r[x][0]*dt, s=abs(vals_r[x][2]), c="tab:blue", zorder=2)
    ax[1].set_xscale("log")
    ax[1].set_title("Rayleigh")

for a in ax:
    a.set_ylim(-150*dt, 150*dt)
    a.set_xlabel("Frequency (Hz)")
    a.set_ylabel("Lag (s)")
    a.grid(which="both")
    a.minorticks_on()
    a.plot(a.get_xlim(), [0, 0], "k", ls="--")

plt.show()